In [1]:
!pip install cdsapi xarray netCDF4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 15.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.8/22.8 MB 27.2 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 23.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 37.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [cdsapi] 7/10 [netCDF4]


In [2]:
import os
import cdsapi
import xarray as xr
import pandas as pd

# Inicializar cliente de Copernicus
c = cdsapi.Client()

# ==============================================================================
# CONFIGURACIÓN DE PARÁMETROS
# ==============================================================================
YEARS = [str(y) for y in range(2018, 2025)]
MONTHS = [f"{m:02d}" for m in range(1, 13)]
DAYS = [f"{d:02d}" for d in range(1, 32)]
TIMES = [f"{h:02d}:00" for h in range(24)]

# Coordenadas (Norte, Oeste, Sur, Este)
LAT_LAND, LON_LAND = 21.0, -86.9
AREA_LAND = [LAT_LAND + 0.2, LON_LAND - 0.2, LAT_LAND - 0.2, LON_LAND + 0.2]

LAT_OCEAN, LON_OCEAN = 21.0, -86.75
AREA_OCEAN = [LAT_OCEAN + 0.5, LON_OCEAN - 0.5, LAT_OCEAN - 0.5, LON_OCEAN + 0.5]

# Directorio para guardar NetCDFs temporales
os.makedirs("data_era5", exist_ok=True)

# Variables
VARS_GROUP_A = [
    '2m_temperature', '2m_dewpoint_temperature', '10m_u_component_of_wind',
    '10m_v_component_of_wind', 'soil_temperature_level_1',
    'soil_temperature_level_2', 'soil_temperature_level_3'
]

VARS_GROUP_B = [
    'total_precipitation', 'surface_solar_radiation_downwards'
]

VARS_GROUP_C = [
    'significant_height_of_combined_wind_waves_and_swell',
    'mean_wave_direction', 'mean_wave_period',
    'significant_height_of_total_swell'
]

# ==============================================================================
# FUNCIONES AUXILIARES
# ==============================================================================
def download_era5(dataset, variables, year, month, area, filename):
    """Descarga de datos asegurando el formato correcto."""
    if os.path.exists(filename) and os.path.getsize(filename) > 2000:
        print(f"  -> El archivo {filename} ya existe. Omitiendo descarga.")
        return

    if os.path.exists(filename):
        os.remove(filename)

    print(f"  -> Descargando {filename}...")

    request_params = {
        'variable': variables,
        'year': year,
        'month': month,
        'day': DAYS,
        'time': TIMES,
        'area': area,
        'data_format': 'netcdf',
        'download_format': 'unarchived' # <--- GARANTIZA QUE SEA NETCDF PURO
    }

    if dataset == 'reanalysis-era5-single-levels':
        request_params['product_type'] = 'reanalysis'

    c.retrieve(dataset, request_params, filename)

def handle_expver(ds):
    """Maneja la dimensión 'expver' (datos preliminares) si existe."""
    if 'expver' in ds.dims:
        ds_1 = ds.sel(expver=1).drop_vars('expver', errors='ignore')
        ds_5 = ds.sel(expver=5).drop_vars('expver', errors='ignore')
        return ds_1.combine_first(ds_5)
    return ds

def clean_dataframe(df):
    """Limpia columnas innecesarias de Xarray."""
    cols_to_drop = ['latitude', 'longitude', 'number', 'step', 'surface', 'valid_time', 'depth']
    return df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# ==============================================================================
# BUCLE PRINCIPAL (MES A MES)
# ==============================================================================
all_data = []

for year in YEARS:
    print(f"\n========== PROCESANDO AÑO {year} ==========")
    for month in MONTHS:
        print(f"\n--- Mes: {month}/{year} ---")

        file_a = f"data_era5/groupA_{year}_{month}.nc"
        file_b = f"data_era5/groupB_{year}_{month}.nc"
        file_c = f"data_era5/groupC_{year}_{month}.nc"

        # 1. Descargas mensuales
        download_era5('reanalysis-era5-land', VARS_GROUP_A, year, month, AREA_LAND, file_a)
        download_era5('reanalysis-era5-land', VARS_GROUP_B, year, month, AREA_LAND, file_b)
        download_era5('reanalysis-era5-single-levels', VARS_GROUP_C, year, month, AREA_OCEAN, file_c)

        # 2. Lectura de los NetCDF del mes
        print("  -> Abriendo archivos NetCDF...")
        ds_a = handle_expver(xr.open_dataset(file_a))
        ds_b = handle_expver(xr.open_dataset(file_b))
        ds_c = handle_expver(xr.open_dataset(file_c))

        # 3. Extraer punto exacto
        pt_a = ds_a.sel(latitude=LAT_LAND, longitude=LON_LAND, method='nearest')
        pt_b = ds_b.sel(latitude=LAT_LAND, longitude=LON_LAND, method='nearest')
        pt_c = ds_c.sel(latitude=LAT_OCEAN, longitude=LON_OCEAN, method='nearest')

        # 4. Convertir a DataFrames
        df_a = clean_dataframe(pt_a.to_dataframe().reset_index())
        df_b = clean_dataframe(pt_b.to_dataframe().reset_index())
        df_c = clean_dataframe(pt_c.to_dataframe().reset_index())

        # 5. Conversiones Grupo B
        if 'tp' in df_b.columns:
            df_b['tp_mm'] = df_b['tp'] * 1000.0
            df_b = df_b.drop(columns=['tp'])

        if 'ssrd' in df_b.columns:
            df_b['ssrd_wm2'] = df_b['ssrd'] / 3600.0
            df_b = df_b.drop(columns=['ssrd'])

        # 6. Merge mensual
        df_merged = pd.merge(df_a, df_b, on='time', how='inner')
        df_merged = pd.merge(df_merged, df_c, on='time', how='inner')

        all_data.append(df_merged)

        # Liberar memoria cerrando los archivos
        ds_a.close()
        ds_b.close()
        ds_c.close()

# ==============================================================================
# EXPORTACIÓN FINAL
# ==============================================================================
print("\nUnificando todos los meses...")
df_final = pd.concat(all_data, ignore_index=True)

# Eliminar posibles filas duplicadas (a veces ocurre en cruces de año bisiesto) y ordenar
df_final = df_final.drop_duplicates(subset=['time']).sort_values('time').reset_index(drop=True)

output_csv = "serie_tiempo_campamento_tamul.csv"
df_final.to_csv(output_csv, index=False)
print(f"¡Proceso finalizado con éxito! Los datos limpios se guardaron en: {output_csv}")


========== PROCESANDO AÑO 2018 ==========

--- Mes: 01/2018 ---
  -> Descargando data_era5/groupA_2018_01.nc...


HTTPError: 403 Client Error: Forbidden for url: https://cds.climate.copernicus.eu/api/retrieve/v1/processes/reanalysis-era5-land/execution
required licences not accepted
Not all the required licences have been accepted; please visit https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land?tab=download#manage-licences to accept the required licence(s).

In [13]:
df_b.shape

(744, 3)